# Übung 20 — Datenbeschaffung, Aufbereitung & EDA Schritt für Schritt zum Prüfungsniveau AP02 (Übung)

Dieses Notebook führt dich in **9 Stufen** zum Niveau der Prüfung
*AP02 Datenbeschaffung, Datenaufbereitung & explorative Datenanalyse*:

| Stufe | Thema | Prüfungsteil |
|---|---|---|
| 1 | BeautifulSoup-Grundlagen an einem Mini-HTML | C |
| 2 | Produktlisten scrapen, Duplikate erkennen | B |
| 3 | Rohtext mit Regex in Zahlen umwandeln | B |
| 4 | Fehlende Werte: zählen, MCAR / MAR / MNAR | A |
| 5 | EDA: Verteilung, Lagemasse, Ausreisser | A, B |
| 6 | Klassen bilden: `cut`, `qcut`, `crosstab` | C |
| 7 | JSON, `merge`, Korrelation, Streudiagramm | A, C |
| 8 | Tabellen von einer Detailseite extrahieren | C |
| 9 | Konzeptfragen mit Bezug zu den Daten | A |

Danach bist du bereit für die **Probeprüfung** `21_probepruefung_ap02.ipynb`.

Ergänze die Zellen mit `# TODO`. Die Zellen **Selbstkontrolle** prüfen dein Ergebnis.

## Hinweise zu den Daten

Alle Daten stammen vom **fiktiven** Schweizer Online-Shop **«VeloMarkt»** und liegen lokal in `Data/velomarkt/`.
Es ist kein Internetzugriff nötig. Spaltenbeschreibungen: `Data/velomarkt/README.md`.

| Datei | Format | Inhalt |
|---|---|---|
| `html/shop_page_1.html` … `shop_page_3.html` | HTML | gespeicherte Produktlisten-Seiten (Paginierung) |
| `html/product_detail.html` | HTML | Detailseite eines E-Bikes (Preisverlauf, technische Daten, Lieferung) |
| `reviews.json` | JSON | gespeicherte Antwort der Reviews-API: Bewertungen pro `product_id` |
| `velos_clean.csv` | CSV | aufbereitete Shop-Daten — **inhaltlich nicht geprüft** |

## Setup

In [ ]:
# Libraries
import os
import re
import json
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from bs4 import BeautifulSoup

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

# Daten von «VeloMarkt» liegen hier:
DATA = 'Data/velomarkt'
print(os.getcwd())

---
## Stufe 1 — BeautifulSoup-Grundlagen

Das folgende Mini-HTML hat denselben Aufbau wie die Shop-Seiten. Öffne zum Vergleich auch
`Data/velomarkt/html/shop_page_1.html` im Editor.

In [ ]:
mini_html = """
<p class="result-count">2 Artikel</p>
<div class="product-card" data-product-id="VM-1">
  <span class="brand">Cube</span> <span class="model">Hyde</span>
  <li class="weight">14,2\xa0kg</li>
  <span class="price">CHF\xa0899.–</span>
</div>
<div class="product-card" data-product-id="VM-2">
  <span class="brand">Stromer</span> <span class="model">ST5</span>
  <li class="weight">27,9\xa0kg</li>
  <span class="price">CHF\xa011'990.–</span>
</div>
"""
mini = BeautifulSoup(mini_html, 'html.parser')

### 1.1 Elemente finden
1. Wie viele Produktkarten (`div.product-card`) gibt es? → `anzahl_karten`
2. Erstelle eine Liste `ids` mit dem Attribut `data-product-id` jeder Karte.
3. Erstelle eine Liste `marken` mit dem Text aller `span.brand`.

In [ ]:
anzahl_karten = None   # TODO
ids = None             # TODO
marken = None          # TODO
print(anzahl_karten, ids, marken)

In [ ]:
# Selbstkontrolle 1.1
assert anzahl_karten == 2 and ids == ['VM-1', 'VM-2'] and marken == ['Cube', 'Stromer']
print('1.1 richtig ✔')

### 1.2 Text sauber auslesen
Lies den Preistext jeder Karte aus. Gib den Text einmal **roh** aus (`repr()` zeigt unsichtbare Zeichen) und
erstelle dann die Liste `preise_text`, in der das geschützte Leerzeichen `\xa0` entfernt und der Text mit
`.strip()` bereinigt ist.

In [ ]:
# TODO: roher Text mit repr()

preise_text = None   # TODO
preise_text

In [ ]:
# Selbstkontrolle 1.2
assert preise_text == ['CHF899.–', "CHF11'990.–"]
print('1.2 richtig ✔')

### 1.3 `select` vs. `select_one`
`select()` liefert **immer eine Liste** (evtl. leer), `select_one()` das **erste** passende Element oder `None`.
Lies mit `select_one` den Text von `p.result-count` und wandle die Zahl darin in einen `int` um → `artikel_total`.

In [ ]:
artikel_total = None   # TODO
artikel_total

In [ ]:
# Selbstkontrolle 1.3
assert artikel_total == 2
print('1.3 richtig ✔')

---
## Stufe 2 — Produktlisten scrapen und Duplikate erkennen

### 2.1 Wie viele Artikel führt der Shop?
Lies aus `shop_page_1.html` die Angabe in `p.result-count` als Zahl → `artikel_shop`.

In [ ]:
artikel_shop = None   # TODO
artikel_shop

In [ ]:
# Selbstkontrolle 2.1
assert artikel_shop == 120
print('2.1 richtig ✔')

### 2.2 Alle drei Seiten scrapen → `df_raw`
Der Code ist vorgegeben (gleicher Aufbau wie in der Prüfung). Führe ihn aus und vergleiche die Anzahl Zeilen mit
`artikel_shop`.

In [ ]:
# Initialize list for the extracted rows
rows = []

# Loop over the three listing pages
for page in [1, 2, 3]:

    # Read html from file
    with open(f'{DATA}/html/shop_page_{page}.html', 'r', encoding='utf-8') as file:
        html_content = file.read()

    # Parse the HTML content
    soup = BeautifulSoup(html_content, 'html.parser')

    # Locate the product cards and extract values
    for card in soup.select('div.product-card'):
        row_data = [card['data-product-id'],
                    card.select('span.brand')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.model')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.category')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.weight')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.battery')[0].get_text().replace('\xa0', '').strip(),
                    card.select('li.range')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.price')[0].get_text().replace('\xa0', '').strip(),
                    card.select('span.availability')[0].get_text().replace('\xa0', '').strip()]
        rows.append(row_data)

# Create a DataFrame
df_raw = pd.DataFrame(rows, columns=['product_id', 'brand', 'model', 'category', 'weight_raw',
                                     'battery_raw', 'range_raw', 'price_raw', 'availability'])

# Show dimensions and first rows
print('Number of rows and columns:', df_raw.shape)
df_raw.head()

### 2.3 Duplikate finden und entfernen
1. Wie viele `product_id` kommen mehrfach vor? (`duplicated()`) → `anzahl_duplikate`
2. Welche IDs sind betroffen? → Liste `doppelte_ids` (sortiert)
3. Entferne die Duplikate anhand von `product_id` und setze den Index neu (`reset_index(drop=True)`) → `df_unique`

*Frage:* Woher kommen die Duplikate? Schau dir an, auf welchen Seiten die betroffenen IDs stehen.

In [ ]:
anzahl_duplikate = None   # TODO
doppelte_ids = None       # TODO
df_unique = None          # TODO
print(anzahl_duplikate, doppelte_ids, df_unique.shape)

In [ ]:
# Selbstkontrolle 2.3
assert anzahl_duplikate == 6 and len(doppelte_ids) == 6
assert df_unique.shape[0] == artikel_shop and df_unique.index.tolist() == list(range(120))
print('2.3 richtig ✔')

---
## Stufe 3 — Rohtext mit Regex in Zahlen umwandeln

### 3.1 Was liefert `re.findall`?
Sage zuerst voraus, was herauskommt, und prüfe dann:

In [ ]:
print(re.findall(r'\d+', "CHF899.–"))
print(re.findall(r'\d+', "CHF1'249.–"))
print(re.findall(r'\d+', "Preis auf Anfrage"))
print(re.findall(r'\d+,\d+|\d+', "25,4kg"))

**Erklärung:**

> [Hier eintragen]

### 3.2 Eine robuste Umwandlungsfunktion
Schreibe die Funktion `preis_zu_zahl(text)`: Sie entfernt den Apostroph, sucht die erste Ziffernfolge und gibt sie als
`int` zurück — oder `None`, wenn keine Zahl vorhanden ist.

In [ ]:
def preis_zu_zahl(text):
    # TODO
    pass

In [ ]:
# Selbstkontrolle 3.2
assert preis_zu_zahl("CHF899.–") == 899
assert preis_zu_zahl("CHF11'990.–") == 11990
assert preis_zu_zahl("Preis auf Anfrage") is None
print('3.2 richtig ✔')

### 3.3 Preise umwandeln — und die Index-Falle
1. Wende `preis_zu_zahl` auf `df_unique['price_raw']` an und speichere das Ergebnis als Spalte `price_chf`
   (Datentyp `Int64`, der auch fehlende Werte erlaubt) → `df_33`.
2. Berechne Anzahl fehlender Preise und den Median.

**Achtung, Index-Falle:** Weist man einer Spalte eine **neue `pd.Series`** zu, richtet pandas die Werte nach dem
**Index** aus, nicht nach der Position. Hat der DataFrame nach `drop_duplicates()` Lücken im Index (z. B. 0–79, 86–125),
landen Werte in falschen Zeilen und es entstehen NaN. Darum `reset_index(drop=True)` (wie in 2.3) oder direkt
`.apply()` auf der Spalte verwenden.

In [ ]:
df_33 = df_unique.copy()
# TODO

anzahl_ohne_preis = None   # TODO
median_preis = None        # TODO
print(anzahl_ohne_preis, median_preis)

In [ ]:
# Selbstkontrolle 3.3
assert anzahl_ohne_preis == 6 and median_preis == 3698.0
print('3.3 richtig ✔')

### 3.4 Gewicht, Akku und Reichweite
Wandle in `df_33` die Rohspalten in Zahlen um:
- `weight_kg` aus `weight_raw` (z. B. `25,4kg` → 25.4, `k. A.` → NaN) — Komma durch Punkt ersetzen!
- `battery_wh` aus `battery_raw` (z. B. `800Wh` → 800, `–` → NaN)
- `range_km` aus `range_raw` (z. B. `bis 145km` → 145, `–` → NaN)

*Tipp:* `.str.extract(r'(\d+,\d+|\d+)', expand=False)` liefert die erste Zahl als Text (oder NaN),
`pd.to_numeric(..., errors='coerce')` wandelt in Zahlen um.

In [ ]:
# TODO

df_33[['weight_kg', 'battery_wh', 'range_km']].describe()

In [ ]:
# Selbstkontrolle 3.4
assert df_33['weight_kg'].isnull().sum() == 4 and df_33['battery_wh'].isnull().sum() == 59
assert df_33['weight_kg'].max() == 28.5 and df_33['range_km'].max() == 180
print('3.4 richtig ✔')

### 3.5 Aufbereitete Daten gegen die Quelle prüfen
`velos_clean.csv` wurde von jemand anderem aufbereitet und ist «inhaltlich nicht geprüft». Vergleiche die Preise:
Verbinde `df_33[['product_id', 'price_chf']]` mit `df_clean[['product_id', 'price_chf']]` über `product_id`
(Suffixe `_scraped` und `_clean`) und finde alle Produkte, deren Preise sich unterscheiden → `abweichungen`.

In [ ]:
# Import data from csv to pandas dataframe named 'df_clean'
df_clean = pd.read_csv(f'{DATA}/velos_clean.csv')

# Show dimensions and first rows
print('Number of rows and columns:', df_clean.shape)
df_clean.head()

In [ ]:
abweichungen = None   # TODO
abweichungen

In [ ]:
# Selbstkontrolle 3.5
assert abweichungen['product_id'].tolist() == ['VM-30032']
print('3.5 richtig ✔')

---
## Stufe 4 — Fehlende Werte

### 4.1 Wo fehlt was?
Zähle die fehlenden Werte pro Spalte in `df_clean` → Series `fehlend` (nur Spalten mit mindestens einem fehlenden Wert).

In [ ]:
fehlend = None   # TODO
fehlend

In [ ]:
# Selbstkontrolle 4.1
assert fehlend.to_dict() == {'weight_kg': 4, 'battery_wh': 59, 'range_km': 59, 'price_chf': 6}
print('4.1 richtig ✔')

### 4.2 Hängt das Fehlen von etwas ab?
Zähle pro `category`, wie viele Preise fehlen, und wie viele Akku-Angaben fehlen. → DataFrame `fehlend_kat` mit den
Spalten `preis_fehlt` und `akku_fehlt` (Index: `category`).

In [ ]:
fehlend_kat = None   # TODO
fehlend_kat

In [ ]:
# Selbstkontrolle 4.2
assert int(fehlend_kat['preis_fehlt'].sum()) == 6
assert set(fehlend_kat[fehlend_kat['preis_fehlt'] > 0].index) == {'E-Mountainbike', 'S-Pedelec'}
assert int(fehlend_kat.loc['Rennvelo', 'akku_fehlt']) == 13
print('4.2 richtig ✔')

### 4.3 MCAR, MAR oder MNAR?
Zur Erinnerung:
- **MCAR** (missing completely at random): Das Fehlen hängt von nichts ab — reiner Zufall.
- **MAR** (missing at random): Das Fehlen hängt von **anderen, beobachteten** Merkmalen ab.
- **MNAR** (missing not at random): Das Fehlen hängt vom **fehlenden Wert selbst** ab.

Ordne zu und begründe mit den Ergebnissen aus 4.1 / 4.2:

a) `price_chf` fehlt, weil der Shop bei Velos über CHF 9'000 «Preis auf Anfrage» anzeigt.
b) `weight_kg` fehlt bei 4 Velos, weil der Lieferant zufällig einzelne Datenblätter nicht geliefert hat.
c) `battery_wh` fehlt bei 59 Velos — alle ohne Elektromotor.
d) Welchen Effekt hat `dropna()` bei a) auf den berichteten Durchschnittspreis?

**Ihre Antwort:**

> [Hier eintragen]

---
## Stufe 5 — EDA: Verteilung, Lagemasse, Ausreisser

### 5.1 Kennzahlen
Berechne für `df_clean['price_chf']` die Kennzahlen mit `describe()` und die Schiefe mit `skew()` → `kennzahlen`,
`schiefe` (auf 2 Stellen gerundet).

In [ ]:
kennzahlen = None   # TODO
schiefe = None      # TODO
print(kennzahlen, '\nSchiefe:', schiefe)

In [ ]:
# Selbstkontrolle 5.1
assert kennzahlen['count'] == 114 and kennzahlen['50%'] == 3748.0 and kennzahlen['max'] == 30981.0
assert abs(schiefe - 4.54) < 0.02
print('5.1 richtig ✔')

**Interpretation:**

> [Hier eintragen]

### 5.2 Verteilung visualisieren
Erstelle nebeneinander ein **Histogramm** (`sns.histplot`) und einen **Boxplot** (`sns.boxplot`) der Preise — mit
Titel und Achsenbeschriftungen.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
# TODO
plt.show()

### 5.3 Ausreisser mit der IQR-Regel
Ein Wert gilt als Ausreisser, wenn er unter `Q1 − 1.5·IQR` oder über `Q3 + 1.5·IQR` liegt (`IQR = Q3 − Q1`).
Finde alle Ausreisser-Zeilen in `df_clean` (Spalten `product_id`, `brand`, `model`, `price_chf`) → `ausreisser`.

In [ ]:
ausreisser = None   # TODO
ausreisser

In [ ]:
# Selbstkontrolle 5.3
assert ausreisser['product_id'].tolist() == ['VM-30032']
print('5.3 richtig ✔')

### 5.4 Mittelwert oder Median pro Gruppe?
Berechne pro `brand` Mittelwert **und** Median des Preises (gerundet auf 2 Stellen), sortiert nach Mittelwert absteigend
→ `pro_marke`. Was fällt bei «Tour de Suisse» auf?

In [ ]:
pro_marke = None   # TODO
pro_marke

In [ ]:
# Selbstkontrolle 5.4
assert pro_marke.loc['Tour de Suisse', 'mean'] == 4113.33 and pro_marke.loc['Tour de Suisse', 'median'] == 1598.0
print('5.4 richtig ✔')

**Beobachtung:**

> [Hier eintragen]

### 5.5 Ausreisser behandeln
Der Wert 30'981 ist nachweislich ein Tippfehler (Shop: 3'098, siehe 3.5). Korrigiere ihn in einer Kopie `df_55` und
berechne den Mittelwert pro Marke erneut (sortiert, gerundet) → `pro_marke_korr`.

*Alternative:* Ist der richtige Wert unbekannt, setzt man einen nachweislich falschen Wert auf `NaN` (und dokumentiert das).
Den Ausreisser **ungeprüft** zu löschen, ist dagegen keine gute Idee: Er könnte auch korrekt sein.

In [ ]:
df_55 = df_clean.copy()
# TODO

pro_marke_korr = None   # TODO
pro_marke_korr

In [ ]:
# Selbstkontrolle 5.5
assert pro_marke_korr.index[-1] == 'Tour de Suisse'
assert abs(pro_marke_korr['Tour de Suisse'] - 1789.75) < 0.01
print('5.5 richtig ✔')

---
## Stufe 6 — Klassen bilden: `cut`, `qcut`, `crosstab`

### 6.1 Feste Grenzen mit `pd.cut`
Teile die Velos nach `weight_kg` in die Klassen `leicht` (bis 10 kg), `mittel` (über 10 bis 20 kg) und `schwer`
(über 20 bis 30 kg) ein → neue Spalte `gewichtsklasse` in `df_6`. Zähle die Velos pro Klasse → `anzahl_klassen`.

In [ ]:
df_6 = df_clean.copy()
# TODO

anzahl_klassen = None   # TODO
anzahl_klassen

In [ ]:
# Selbstkontrolle 6.1
assert anzahl_klassen.to_dict() == {'leicht': 35, 'mittel': 24, 'schwer': 57}
print('6.1 richtig ✔')

### 6.2 Gleich grosse Gruppen mit `pd.qcut`
Teile die Velos nach `price_chf` in **drei gleich grosse** Preisklassen `günstig`, `mittel`, `teuer` → Spalte
`segment`. Gib die Klassengrenzen aus (`retbins=True`). Was passiert mit Velos ohne Preis?

In [ ]:
# TODO

In [ ]:
# Selbstkontrolle 6.2
assert df_6['segment'].notnull().sum() == 114 and df_6['segment'].isnull().sum() == 6
print('6.2 richtig ✔')

### 6.3 Kontingenztabelle
Erstelle mit `pd.crosstab` die Anzahl Velos pro `category` (Zeilen) und `segment` (Spalten) mit Zeilen- und
Spaltensummen (`margins=True`) → `kreuz`. Erstelle zusätzlich eine Variante mit Zeilenprozenten
(`normalize='index'`, gerundet auf 2 Stellen) → `kreuz_prozent`.

In [ ]:
kreuz = None           # TODO
kreuz_prozent = None   # TODO
display(kreuz, kreuz_prozent)

In [ ]:
# Selbstkontrolle 6.3
assert kreuz.loc['All', 'All'] == 114
assert kreuz.loc['Kindervelo', 'günstig'] == 14
print('6.3 richtig ✔')

---
## Stufe 7 — JSON, `merge` und Korrelation

### 7.1 JSON einlesen
Lies `reviews.json` in den DataFrame `df_reviews` ein.

In [ ]:
df_reviews = None   # TODO
df_reviews.head()

In [ ]:
# Selbstkontrolle 7.1
assert df_reviews.shape == (99, 3)
print('7.1 richtig ✔')

### 7.2 Welcher Join-Typ?
Verbinde `df_clean` mit `df_reviews` über `product_id` mit `how='inner'`, `'left'` und `'outer'`. Speichere jeweils die
Anzahl Zeilen in `n_inner`, `n_left`, `n_outer`. Untersuche mit `indicator=True`, wie viele Bewertungen zu Produkten
gehören, die **nicht mehr im Sortiment** sind → `n_nur_reviews`.

In [ ]:
n_inner = None         # TODO
n_left = None          # TODO
n_outer = None         # TODO
n_nur_reviews = None   # TODO
print(n_inner, n_left, n_outer, n_nur_reviews)

In [ ]:
# Selbstkontrolle 7.2
assert (n_inner, n_left, n_outer, n_nur_reviews) == (95, 120, 124, 4)
print('7.2 richtig ✔')

### 7.3 Pearson oder Spearman?
Berechne für die Velos **mit** Bewertung und **mit** Preis die Korrelation zwischen `price_chf` und `avg_score` —
einmal nach Pearson, einmal nach Spearman (je auf 2 Stellen gerundet) → `r_pearson`, `r_spearman`.

*Tipp:* `df[['price_chf', 'avg_score']].corr(method='spearman')` — fehlende Werte werden paarweise ignoriert.

In [ ]:
df_73 = None       # TODO: inner merge
r_pearson = None   # TODO
r_spearman = None  # TODO
print(r_pearson, r_spearman)

In [ ]:
# Selbstkontrolle 7.3
assert abs(r_pearson - 0.43) < 0.011 and abs(r_spearman - 0.34) < 0.011
print('7.3 richtig ✔')

**Erklärung:**

> [Hier eintragen]

### 7.4 Wie verlässlich ist ein Durchschnitt aus 2 Bewertungen?
Einige Velos haben nur 1–3 Bewertungen. Berechne die Spearman-Korrelation nur für Velos mit **mindestens 5** Bewertungen
und Preis → `r_spearman_5` (gerundet), Anzahl Velos → `n_5`. Zeichne ein Streudiagramm (`sns.scatterplot`) mit Titel
und beschrifteten Achsen.

In [ ]:
# TODO

In [ ]:
# Selbstkontrolle 7.4
assert n_5 == 82 and abs(r_spearman_5 - 0.35) < 0.011
print('7.4 richtig ✔')

---
## Stufe 8 — Tabellen von einer Detailseite

Öffne `Data/velomarkt/html/product_detail.html` im Editor: Die Seite hat **drei** Tabellen mit derselben Klasse
`data-table`. Vor jeder Tabelle steht eine `<h2>`-Überschrift.

### 8.1 Die richtige Tabelle auswählen
1. Wie viele Tabellen gibt es? → `anzahl_tabellen`
2. Wähle die Tabelle **«Technische Daten»** aus, und zwar **über die Überschrift**, nicht über die Position: Suche das
   `h2`, dessen Text «Technische Daten» ist, und nimm die nächste Tabelle (`.find_next('table')`) → `table`

In [ ]:
with open(f'{DATA}/html/product_detail.html', 'r', encoding='utf-8') as file:
    soup_detail = BeautifulSoup(file.read(), 'html.parser')

anzahl_tabellen = None   # TODO
table = None             # TODO

In [ ]:
# Selbstkontrolle 8.1
assert anzahl_tabellen == 3 and 'Panasonic' in table.get_text()
print('8.1 richtig ✔')

### 8.2 Zwischenüberschriften entfernen
Der folgende Code überführt die Tabelle Zeile für Zeile in `df_82`. Schau dir das Ergebnis an: Die
Zwischenüberschriften (z. B. «Antrieb») haben nur **eine** Zelle, darum ist `wert` dort leer (`None`).
Entferne diese Zeilen und setze den Index neu.

In [ ]:
rows = []
for row in table.find_all('tr'):
    rows.append([cell.get_text().strip() for cell in row.find_all('td')])
df_82 = pd.DataFrame(rows, columns=['merkmal', 'wert'])
display(df_82)

# TODO: Zwischenüberschriften entfernen

df_82

In [ ]:
# Selbstkontrolle 8.2
assert len(df_82) == 9 and 'Antrieb' not in df_82['merkmal'].tolist()
print('8.2 richtig ✔')

### 8.3 Preisverlauf auswerten
Lies die Tabelle «Preisverlauf» in `df_83` (Spalten `monat`, `preis_raw`) und ergänze die Spalte `preis_chf` (Zahl).
Um wie viele CHF ist der Preis seit März gesunken? → `rueckgang`

In [ ]:
df_83 = None      # TODO
rueckgang = None  # TODO
df_83

In [ ]:
# Selbstkontrolle 8.3
assert rueckgang == 400
print('8.3 richtig ✔')

---
## Stufe 9 — Konzeptfragen (wie Prüfungsteil A)

Antworten zählen in der Prüfung nur **mit Bezug zu den Daten**. Nenne Spalten, Zahlen und konkrete Beispiele.

### 9.1 Korrelation ist nicht Kausalität
Ein Kollege findet zwischen `battery_wh` und `price_chf` (nur E-Bikes) einen Pearson-Koeffizienten von r = −0.10 und
folgert: *«Die Akkugrösse hat keinen Einfluss auf den Preis.»* Nenne zwei fachliche Einwände.

**Ihre Antwort:**

> [Hier eintragen]

### 9.2 Verteilung beschreiben
Für `weight_kg` liefert pandas: `count 116, mean 17.30, std 7.55, min 6.80, 25% 9.70, 50% 16.40, 75% 24.42, max 28.50`
und `skew() = 0.00`. Eine Kollegin sagt: *«Schiefe 0 — die Gewichte sind schön symmetrisch um 17 kg verteilt.»*
Stimmt das? Welche Grafik hilft?

**Ihre Antwort:**

> [Hier eintragen]

---
**Geschafft!** Weiter mit der Probeprüfung `21_probepruefung_ap02.ipynb` — mit Zeitlimit und ohne Lösung.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
import os
import platform
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')